# R2lab F1 handover between two OCUDU DUs with Benetel RUs (O-RAN 7.2)

# **5G Deployment Example With R2lab:**

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UDM, UDR, UPF
- OCUDU split RAN: CU-CP and CU-UP on the VM `node2`, and 2 DUs on two 100 MHz carriers 200 MHz apart (inter-frequency): `du01` on `sopnode-f3` with the Benetel RAN550 `benetel1`, `du02` on `sopnode-f4` with the Benetel RAN550 `benetel2`, both over the O-RAN 7.2 fronthaul (Open Fronthaul library of OCUDU, DPDK on an SR-IOV VF, PTP time synchronization)
- COTS UEs: `qhat01` and `qhat02` in R2Lab, attached to `du01` one after the other, then `qhat01` is handed over to `du02` with the remote command `trigger_handover` of the CU-CP; `qhat03` stays disconnected
- Core on the VM `node10`, monitoring on the VM `node11`; CSI: `du01` logs SRS channel estimates per RB (OCUDU DU image with the CSI logger), shown live on `node11` and collected per traffic section

Same nodes, RUs and carriers as `oai-benetel-2du`; the OCUDU radio and fronthaul parameters are those of the 5g_ansible OCUDU Benetel setup (https://github.com/turletti/srsran-helm, branch `ocudu`, `charts/srsran-gnb/values-benetel1.yaml` and `values-benetel2.yaml`), with the 2.5 ms TDD pattern of the OAI notebook. No RIC is deployed: the handover is triggered from the CU-CP.

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal servers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
#! slices auth login

Install IFRAN:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# CSI support is in this repository (branch csi): install it in editable mode.
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
# O-RAN 7.2 node setup (K8SManagement.setup_fhi72) is on the IFRAN fhi72 branch. Install it after this
# repository, whose ifran@main requirement would otherwise replace it.
# To test unpushed IFRAN changes, set IFRAN to the path of a local clone. Do not install it in editable
# mode (-e): the IFRAN Ansible roles are only found in an installed package.
IFRAN = "ifran @ git+https://github.com/ifran-project/ifran.git@fhi72"
# IFRAN = "/Users/<user>/git/post-5g/ifran"
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "{IFRAN}"
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


Install the Ansible dependencies:

In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r ../../../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

IFRAN keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

Edit the SLICES project, the VM infrastructure, flavor and lifetime here. Keep the SLICES CLI authenticated in this environment.

In [ ]:
from datetime import UTC, datetime

# VMs of the experiment (BI provisioner); the experiment name includes a timestamp, so that a run never reuses
# (and deletes) the VMs of another one
PROJECT = "post5g-beta"
EXPERIMENT = "ocudu-benetel-2du-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
INFRA = "fr-sophia2-bi-vm1"
VM_FLAVOR = "large"
VM_DURATION = "8h"  # VM lifetime
VM_NODES = ["node1", "node2", "node10", "node11"]
print(f"Experiment: {EXPERIMENT}")

In [ ]:
from metamodel_3gpp.models.metamodel import MetaModel

# Inter-frequency F1 handover: two 100 MHz carriers (273 PRB). On a single carrier, the two RUs in the
# same room interfere: the UEs lose the link and re-establish on either cell, and a handover towards the weaker
# cell fails (the UE does not decode the target RAR).
# du01 keeps the carrier of the 5g_ansible Benetel setup (rru/du-config-benetel.yaml), du02 is 298.08 MHz above
# (207 SSB raster steps, same SSB and CORESET#0 offsets from point A), at the top of n78 (3699.66-3797.94 MHz):
# ~200 MHz between the carriers. With adjacent carriers (2.5 MHz apart), du02 in a bad state after a crash
# jammed the uplink of du01 (2026-11-19: false PRACH detections, out-of-sync UEs). TDD 2.5 ms: 3 DL slots + 6 DL symbols,
# 1 UL slot + 4 UL symbols.
# OCUDU: 4T2R (4 DL antennas, 2 UL antennas, Benetel mimo_mode 1_2_3_4_4x2) and PRACH configuration 159, as in
# the OCUDU RAN550 profile; same carriers and 2.5 ms TDD pattern as the OAI notebook.
metamodel = MetaModel.from_yaml(
"""
rus:
  benetel1:
    type: real
    timeSource: external
    clockSource: external
    numberOfTxPorts: 4
    numberOfRxPorts: 2
    txAttenuationDB: 0
    rxAttenuationDB: 0
    rxGainDB: 75
  benetel2:
    type: real
    timeSource: external
    clockSource: external
    numberOfTxPorts: 4
    numberOfRxPorts: 2
    txAttenuationDB: 0
    rxAttenuationDB: 0
    rxGainDB: 75
ran:
  components:
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
      config:
        cu_cp: cucp
    du01:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 100
          downlinkArfcn: 630048  # carrier centre, used by OCUDU (dl_arfcn)
          pointAFrequencyArfcn: 626772  # 3401.58 MHz, carrier 3401.58-3499.86 MHz
          ssbArfcn: 630048  # 3450.72 MHz
          ssbPbchBlockPowerDbm: -15
          prachConfigurationIndex: 159
          msg1FrequencyStartPrb: 0
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 3
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 1
            nrofUplinkSymbols: 4
        gnbDuId: 1
        nrCellId: 1
        physicalCellId: 1
        ru: benetel1
        cu_cp: cucp
        cu_up: cuup
    du02:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 100
          downlinkArfcn: 649920  # carrier centre, used by OCUDU (dl_arfcn)
          pointAFrequencyArfcn: 646644  # 3699.66 MHz, carrier 3699.66-3797.94 MHz
          ssbArfcn: 649920  # 3748.80 MHz
          ssbPbchBlockPowerDbm: -15
          prachConfigurationIndex: 159
          msg1FrequencyStartPrb: 0
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 3
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 1
            nrofUplinkSymbols: 4
        gnbDuId: 2
        nrCellId: 2
        physicalCellId: 2
        ru: benetel2
        cu_cp: cucp
        cu_up: cuup
ric:
  components: {}
ues:
  qhat01:
    type: cots
    config:
      supi: '001010000000006'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - du01
  qhat02:
    type: cots
    config:
      supi: '001010000000007'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - du01
  qhat03:
    type: cots
    config:
      supi: '001010000000008'
      authenticationData:
        k: fec86ba6eb707ed08905757b1bb44b8f
        opc: C42449363BBAD02B66D16BC975D77CC1
      gnbs:
      - du01
"""
)

To see the full content of this Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

Four VMs (BI provisioner, `INFRA` above): `node1` runs the Kubernetes control plane and the iperf server, `node2` the CU-CP and the CU-UP, `node10` the core and its database, `node11` the monitoring and the live CSI view. `sopnode-f3` and `sopnode-f4`, provisioned by POS, run the DUs (the `legacy_pos` provisioner remains declared but no node uses it). The `fhi72` block of each RAN node describes its O-RAN 7.2 setup: fronthaul PF and its VFs (one per C/U plane, with the DU MAC and the VLAN of its RU: 801 for `benetel1`, 802 for `benetel2`), housekeeping CPUs, NIC IRQ pinning and PTP. Both RAN nodes boot the PREEMPT_RT image with the same boot parameters.

For the radio hardware, we are going to use `benetel1` and `benetel2` as RUs in R2Lab, as well as `qhat01`, `qhat02` and `qhat03` as UEs in R2Lab.

The deployment model needs your [R2Lab](https://r2lab.inria.fr/tuto-010-registration.md) credentials (your R2Lab slice name, email, and password).

A good practice to prevent private information leak (e.g., password) is to store the sensitive information in a special location. In this, we keep it as environment variables that we store in a `.env` [file](https://pypi.org/project/python-dotenv/), next to this notebook or in a parent directory (e.g. the nador root, shared by all the notebooks; `load_dotenv()` takes the first one found going up from the notebook directory):

```
R2LAB_SLICE=<YOUR_R2LAB_SLICE>
R2LAB_EMAIL=<YOUR_R2LAB_EMAIL>
R2LAB_PASSWORD=<YOUR_R2LAB_PASSWORD>
```

The following lines load variables from this `.env` file. Make sure to never commit your secrets in public repositories.

In [ ]:
%pip install python-dotenv

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()
R2LAB_SLICE = os.getenv("R2LAB_SLICE")
R2LAB_EMAIL = os.getenv("R2LAB_EMAIL")
R2LAB_PASSWORD = os.getenv("R2LAB_PASSWORD")
missing = [name for name in ("R2LAB_SLICE", "R2LAB_EMAIL", "R2LAB_PASSWORD") if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Set {missing} in a .env file next to this notebook or in a parent directory")

We can now create the deployment model.

In [ ]:
from pathlib import Path

from ifran.models.deployment import DeploymentModel

# OCUDU images: IFRAN defaults for a Benetel RU (registry.gitlab.com/ocudu/ocudu/images: du-dpdk or gnb-dpdk, cu-cp,
# cu-up); enable_csi replaces the DU image by the CSI logger build (r2labuser/ocudu-du-csi:release_26_10).

# VMs (BI provisioner): Kubernetes control plane and iperf server (node1), CU-CP and CU-UP (node2), core and
# database (node10), monitoring and live CSI view (node11). sopnode-f3 and sopnode-f4 (POS) run the DUs.
deployment_model = DeploymentModel.from_yaml(
f"""
params:
  provisioning:
    extra_path: ''
  k8s:
    master_node: node1
    local_kubeconfig_path: {Path(f".k8s-{EXPERIMENT}.conf").resolve()}
provisioners:
  bi:
    type: bi
    params:
      project: {PROJECT}
      experiment: {EXPERIMENT}
      infra: {INFRA}
      flavor: {VM_FLAVOR}
      create_experiment: true
      node_duration: {VM_DURATION}
  pos:
    type: pos
    params:
      attempt_reservation: false
  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <YOUR_SLICES_USERNAME>
  r2lab:
    type: r2lab
    params:
      r2lab_slice: {json.dumps(R2LAB_SLICE)}
      reservation_params:
        email: {json.dumps(R2LAB_EMAIL)}
        password: {json.dumps(R2LAB_PASSWORD)}
nodes:
  node1:
    provisioner: bi
    roles: []
  node2:
    provisioner: bi
    roles:
    - ran
  node10:
    provisioner: bi
    roles:
    - core
    - database
  node11:
    provisioner: bi
    roles: []
  sopnode-f3:
    provisioner: pos
    roles:
    - ran
    provisioning_params:
      # PREEMPT_RT kernel image, as with 5g_ansible (pos_rt_image)
      image: ubuntu-questing-rt@2025-10-29T19:34:30+00:00
      is_staged_image: true
      # 5g_ansible boot_parameters_rt_f3: housekeeping CPUs 0-5 (= fhi72 reserved_cpus), 40 x 1G hugepages
      boot_params: net.ifnames=1 biosdevname=0 isolcpus=managed_irq,6-63 nohz_full=6-63
        nohz=on rcu_nocbs=6-63 kthread_cpus=0-5 irqaffinity=0-5 rcu_nocb_poll
        selinux=0 enforcing=0 intel_pstate=disable nosoftlockup
        hugepagesz=1G hugepages=40 hugepagesz=2M hugepages=0 default_hugepagesz=1G
        mitigations=off intel_iommu=on processor.max_cstate=1 idle=poll
        intel_idle.max_cstate=0 iommu=pt skew_tick=1 tsc=nowatchdog
        nmi_watchdog=0 softlockup_panic=0 audit=0 mce=off nosmt
    fhi72:
      fronthaul_interface: ens15f1np1
      # DU MAC addresses and VLAN expected by the RU (also loaded in the RU by ran.configure_rus())
      vfs:
      - resource_name: ruvfiou
        plane: U
        mac: 00:11:22:33:44:66
        vlan: 801
      - resource_name: ruvfioc
        plane: C
        mac: 00:11:22:33:44:67
        vlan: 801
      reserved_cpus: 0-5
      irq_pattern: ice-|ens15f[0-3]np[0-3]
      ptp:
        interface: ens15f2np2
  sopnode-f4:
    provisioner: pos
    roles:
    - ran
    provisioning_params:
      # PREEMPT_RT kernel image, as with 5g_ansible (pos_rt_image)
      image: ubuntu-questing-rt@2025-10-29T19:34:30+00:00
      is_staged_image: true
      # 5g_ansible boot_parameters_rt_f3 (same hardware as sopnode-f3): housekeeping CPUs 0-5 (= fhi72 reserved_cpus), 40 x 1G hugepages
      boot_params: net.ifnames=1 biosdevname=0 isolcpus=managed_irq,6-63 nohz_full=6-63
        nohz=on rcu_nocbs=6-63 kthread_cpus=0-5 irqaffinity=0-5 rcu_nocb_poll
        selinux=0 enforcing=0 intel_pstate=disable nosoftlockup
        hugepagesz=1G hugepages=40 hugepagesz=2M hugepages=0 default_hugepagesz=1G
        mitigations=off intel_iommu=on processor.max_cstate=1 idle=poll
        intel_idle.max_cstate=0 iommu=pt skew_tick=1 tsc=nowatchdog
        nmi_watchdog=0 softlockup_panic=0 audit=0 mce=off nosmt
    fhi72:
      fronthaul_interface: ens15f2np2  # sw5 Ethernet1/37
      # DU MAC addresses and VLAN expected by the RU (also loaded in the RU by ran.configure_rus())
      vfs:
      - resource_name: ruvfiou
        plane: U
        mac: 00:11:22:33:44:66
        vlan: 802
      - resource_name: ruvfioc
        plane: C
        mac: 00:11:22:33:44:67
        vlan: 802
      reserved_cpus: 0-5
      irq_pattern: ice-|ens15f[0-3]np[0-3]
      ptp:
        interface: ens15f3np3
ran:
  implementation: ocudu
  components:
    # CU-CP and CU-UP on a VM: the CPUs of the RAN nodes are isolated for the DUs
    cucp:
      type: cu_cp
      node_name: node2
      advanced_config:
        # release an inactive UE after 2 h (maximum) instead of 120 s: an idle UE attaches again with a new C-RNTI,
        # which the handover command names
        inactivity_timer: 7200
        # remote control server (WebSocket): handover command trigger_handover of nador.handover.run_handover
        remote_control:
          enabled: true
          bind_addr: 0.0.0.0
          port: 8001
    cuup:
      type: cu_up
      node_name: node2
    du01:
      type: du
      node_name: sopnode-f3
      advanced_config:
        # gNB-DU ID and sector of the cell (NCI = gNB ID + sector): unique per DU, or the CU-CP rejects the F1 setup
        # of the second DU ("Duplicate DU ID", then "Duplicate served cell CGI")
        gnb_du_id: 1
        sector_id: 1
        # Open Fronthaul (O-RAN 7.2) to the Benetel RAN550, as in the 5g_ansible OCUDU Benetel setup (srsran-helm
        # charts/srsran-gnb/values-benetel1.yaml): one VF (U-plane VF of the node, MAC 00:11:22:33:44:66) for both planes.
        # @FHI72_...@ placeholders are replaced by the nador chart when the pod starts (VF PCI address, pinned CPUs).
        ru_ofh:
          t1a_max_cp_dl: 470
          t1a_min_cp_dl: 419
          t1a_max_cp_ul: 336
          t1a_min_cp_ul: 285
          t1a_max_up: 345
          t1a_min_up: 294
          ta4_max: 200
          ta4_min: 0
          is_prach_cp_enabled: true
          compr_method_ul: bfp
          compr_bitwidth_ul: 9
          compr_method_dl: bfp
          compr_bitwidth_dl: 9
          compr_method_prach: bfp
          compr_bitwidth_prach: 9
          enable_ul_static_compr_hdr: false
          enable_dl_static_compr_hdr: false
          iq_scaling: 5.5
          cells:
          - network_interface: "@FHI72_PCI@"
            mtu: 9216
            ru_mac_addr: 8c:1f:64:d1:12:8c
            du_mac_addr: 00:11:22:33:44:66
            vlan_tag_cp: 801
            vlan_tag_up: 801
            prach_port_id: [4, 5]
            dl_port_id: [0, 1, 2, 3]
            ul_port_id: [0, 1]
        prach:
          prach_config_index: 159
          prach_root_sequence_index: 1
          zero_correlation_zone: 0
          # contention-free random access (handover): preambles 60-63 are kept out of contention-based access
          cfra_enabled: true
          total_nof_ra_preambles: 64
          nof_cb_preambles_per_ssb: 60
        # uplink power of the 5g_ansible setup: closed loop, so that the UEs near the RU do not saturate its input
        pucch:
          p0_nominal: -90
          enable_cl_loop_pw_control: true
        pusch:
          p0_nominal_with_grant: -76
          enable_cl_loop_pw_control: true
          target_sinr: 15
          max_rank: 1
        # PHY at info writes a line per PUCCH/PDSCH/PUSCH (99% of the log): kubelet then rotates the 10 MB log files
        # within seconds, and the handover analysis loses the MAC lines it reads (PRACH, UE creation)
        log:
          filename: stdout
          all_level: info
          phy_level: warning
          config_level: info
        # remote control server and JSON metrics, as in 5g_ansible: the metrics-exporter sidecar of the nador chart
        # subscribes to them and exports the ue_* metrics of the RAN panels (srsRAN dashboard of the monitoring)
        remote_control:
          enabled: true
          bind_addr: 0.0.0.0
          port: 8001
        metrics:
          enable_json: true
          autostart_stdout_metrics: true
          layers:
            enable_app_usage: true
            enable_sched: true
            enable_rlc: true
            enable_mac: true
            enable_du_low: true
            enable_ru: true
        # threads on the CPUs pinned by the static CPU manager: CPU 0 for DPDK (hal eal_args, set by nador), 1 for the
        # OFH timing, 2 for the OFH Tx/Rx, 3 to 5 for the main pool
        expert_execution:
          affinities:
            main_pool_cpus: "@FHI72_CPU_3@,@FHI72_CPU_4@,@FHI72_CPU_5@"
            main_pool_pinning: mask
            ofh:
              timing_cpu: "@FHI72_CPU_1@"
              txrx_cpus: "@FHI72_CPU_2@"
          threads:
            main_pool:
              nof_threads: 3
              task_queue_size: 4096
              backoff_period: 50
            upper_phy:
              pdsch_processor_type: auto
              pdsch_cb_batch_length: 4
              max_pusch_and_srs_concurrency: 2
              max_pucch_concurrency: 2
              max_pdsch_concurrency: 2
            lower_phy:
              execution_profile: triple
      # DPDK: privileged container (VF bound to vfio-pci, hugepages)
      security_context:
        privileged: true
        capabilities:
          add: [IPC_LOCK, SYS_NICE, NET_ADMIN]
      # Guaranteed pod (requests = limits, integer cpu): the static CPU manager pins its CPUs
      resources:
        define: true
        requests:
          nf:
            cpu: 8
            memory: 8Gi
            hugepages_1gi: 2Gi
        limits:
          nf:
            cpu: 8
            memory: 8Gi
            hugepages_1gi: 2Gi
    du02:
      type: du
      node_name: sopnode-f4
      advanced_config:
        # gNB-DU ID and sector of the cell: unique per DU (see du01)
        gnb_du_id: 2
        sector_id: 2
        # Open Fronthaul (O-RAN 7.2) to the Benetel RAN550, as in the 5g_ansible OCUDU Benetel setup (srsran-helm
        # charts/srsran-gnb/values-benetel2.yaml): one VF (U-plane VF of the node, MAC 00:11:22:33:44:66) for both planes.
        # @FHI72_...@ placeholders are replaced by the nador chart when the pod starts (VF PCI address, pinned CPUs).
        ru_ofh:
          t1a_max_cp_dl: 470
          t1a_min_cp_dl: 419
          t1a_max_cp_ul: 336
          t1a_min_cp_ul: 285
          t1a_max_up: 345
          t1a_min_up: 294
          ta4_max: 200
          ta4_min: 0
          is_prach_cp_enabled: true
          compr_method_ul: bfp
          compr_bitwidth_ul: 9
          compr_method_dl: bfp
          compr_bitwidth_dl: 9
          compr_method_prach: bfp
          compr_bitwidth_prach: 9
          enable_ul_static_compr_hdr: false
          enable_dl_static_compr_hdr: false
          iq_scaling: 5.5
          cells:
          - network_interface: "@FHI72_PCI@"
            mtu: 9216
            ru_mac_addr: 8c:1f:64:d1:12:50
            du_mac_addr: 00:11:22:33:44:66
            vlan_tag_cp: 802
            vlan_tag_up: 802
            prach_port_id: [4, 5]
            dl_port_id: [0, 1, 2, 3]
            ul_port_id: [0, 1]
        prach:
          prach_config_index: 159
          prach_root_sequence_index: 65
          zero_correlation_zone: 0
          # contention-free random access (handover): preambles 60-63 are kept out of contention-based access
          cfra_enabled: true
          total_nof_ra_preambles: 64
          nof_cb_preambles_per_ssb: 60
        # uplink power of the 5g_ansible setup: closed loop, so that the UEs near the RU do not saturate its input
        pucch:
          p0_nominal: -90
          enable_cl_loop_pw_control: true
        pusch:
          p0_nominal_with_grant: -76
          enable_cl_loop_pw_control: true
          target_sinr: 15
          max_rank: 1
          # du02 only: QPSK uplink (MCS 0 to 9 of the qam64 table). On 2026-11-20 the PUSCH of qhat01 on du02 failed
          # from 16QAM up (64QAM: 48 OK / 779 KO, 256QAM: 0 / 240) at 22-28 dB SINR, while du01 decoded 16QAM to
          # 256QAM: the UE was released after the handover. Cause not found yet (benetel2, carrier at 3748.8 MHz or
          # sopnode-f4); remove these two lines to test it.
          mcs_table: qam64
          max_ue_mcs: 9
        # PHY at info writes a line per PUCCH/PDSCH/PUSCH (99% of the log): kubelet then rotates the 10 MB log files
        # within seconds, and the handover analysis loses the MAC lines it reads (PRACH, UE creation)
        log:
          filename: stdout
          all_level: info
          phy_level: warning
          config_level: info
        # remote control server and JSON metrics, as in 5g_ansible: the metrics-exporter sidecar of the nador chart
        # subscribes to them and exports the ue_* metrics of the RAN panels (srsRAN dashboard of the monitoring)
        remote_control:
          enabled: true
          bind_addr: 0.0.0.0
          port: 8001
        metrics:
          enable_json: true
          autostart_stdout_metrics: true
          layers:
            enable_app_usage: true
            enable_sched: true
            enable_rlc: true
            enable_mac: true
            enable_du_low: true
            enable_ru: true
        # threads on the CPUs pinned by the static CPU manager: CPU 0 for DPDK (hal eal_args, set by nador), 1 for the
        # OFH timing, 2 for the OFH Tx/Rx, 3 to 5 for the main pool
        expert_execution:
          affinities:
            main_pool_cpus: "@FHI72_CPU_3@,@FHI72_CPU_4@,@FHI72_CPU_5@"
            main_pool_pinning: mask
            ofh:
              timing_cpu: "@FHI72_CPU_1@"
              txrx_cpus: "@FHI72_CPU_2@"
          threads:
            main_pool:
              nof_threads: 3
              task_queue_size: 4096
              backoff_period: 50
            upper_phy:
              pdsch_processor_type: auto
              pdsch_cb_batch_length: 4
              max_pusch_and_srs_concurrency: 2
              max_pucch_concurrency: 2
              max_pdsch_concurrency: 2
            lower_phy:
              execution_profile: triple
      # DPDK: privileged container (VF bound to vfio-pci, hugepages)
      security_context:
        privileged: true
        capabilities:
          add: [IPC_LOCK, SYS_NICE, NET_ADMIN]
      # Guaranteed pod (requests = limits, integer cpu): the static CPU manager pins its CPUs
      resources:
        define: true
        requests:
          nf:
            cpu: 8
            memory: 8Gi
            hugepages_1gi: 2Gi
        limits:
          nf:
            cpu: 8
            memory: 8Gi
            hugepages_1gi: 2Gi
rus:
  benetel1:
    vendor: benetel
    model: ran550
    provisioner: r2lab
    ips:
    - 192.168.233.101
    # ru_config.cfg of the 5g_ansible OCUDU profile (ru_profiles/benetel/config/benetel1-srsran-...cfg):
    # dynamic compression, no long-form PRACH compression, and the C-plane sent to the single DU VF
    benetel:
      tx_power_dbm: 24
      downlink_scaling: 0
      compression: dynamic_compressed
      extra:
        c_plane_du_mac: 00:11:22:33:44:66
        lf_prach_compression_enable: 'false'
  benetel2:
    vendor: benetel
    model: ran550
    provisioner: r2lab
    ips:
    - 192.168.233.102
    # ru_config.cfg of the 5g_ansible OCUDU profile (ru_profiles/benetel/config/benetel1-srsran-...cfg):
    # dynamic compression, no long-form PRACH compression, and the C-plane sent to the single DU VF
    benetel:
      tx_power_dbm: 24
      downlink_scaling: 0
      compression: dynamic_compressed
      extra:
        c_plane_du_mac: 00:11:22:33:44:66
        lf_prach_compression_enable: 'false'
ues:
  qhat01:
    type: cots
    provisioner: r2lab
  qhat02:
    type: cots
    provisioner: r2lab
  qhat03:
    type: cots
    provisioner: r2lab
"""
)

> *IMPORTANT NOTE*: Make sure your R2Lab credentials are not shared publicly!

By default, the metamodel and deployment model will use Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
and one 5G slice (`internet`) with one DNN (`internet`).

## **1.**   Provisioning nodes:

In [ ]:
from ifran.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision_nodes()

## **2.**   Kubernetes deployment:

In [ ]:
from ifran.deployment import K8SManagement

k8s = K8SManagement(prov)

In [ ]:
await k8s.deploy()

### R2Lab netplan of sopnode-f3 and sopnode-f4

As with 5g_ansible: DHCP on the PTP interface (`ptp`, which also carries the RU management network 192.168.233.0/24) gives `sopnode-f3` and `sopnode-f4` access to the RUs (`ssh benetel1`, `ssh benetel2`), and the USRP and AW2S VLANs on the fronthaul PF (`rrus`). Only the netplan role of the r2lab example tuning is applied: the fhi72 setup below replaces its NIC and CPU tuning. Apply it before the fhi72 setup, as `netplan apply` resets the interfaces.

In [ ]:
from pathlib import Path

from ifran.deployment import MiscellaneousManagement

misc = MiscellaneousManagement(k8s)
await misc.run_playbook(
    playbook=str(Path("netplan.yaml").resolve()),
    roles_paths=[str(Path("../oai-aw2s/tuning/ansible/roles").resolve())],
)

### O-RAN 7.2 fronthaul setup of sopnode-f3 and sopnode-f4

Once the RAN nodes are in the cluster: VFs created on the fronthaul PF (`ens15f1np1` on `sopnode-f3`, `ens15f2np2` on `sopnode-f4`) and bound to `vfio-pci`, NIC IRQs pinned on the housekeeping CPUs, kubelet static CPU manager (`full-pcpus-only`), SR-IOV device plugin (resources `openshift.io/ruvfioc` and `openshift.io/ruvfiou`), and PTP (`ptp4l` slave on `ens15f2np2` / `ens15f3np3`, domain 24, and `phc2sys`).

In [ ]:
await k8s.setup_fhi72()


Check each RAN node: 40 free 1G hugepages, both VFs with their MAC and VLAN (801 on `sopnode-f3`, 802 on `sopnode-f4`) bound to `vfio-pci`, `ptp4l` and `phc2sys` active, and a PTP grandmaster present with a small `master_offset` (ns).

In [ ]:
# Fronthaul node state: 1G hugepages, VFs (MAC/VLAN) bound to vfio-pci, PTP (ptp4l slave, phc2sys)
FHI72_NODES = {"sopnode-f3": "ens15f1np1", "sopnode-f4": "ens15f2np2"}  # RAN node: fronthaul PF
for node, pf in FHI72_NODES.items():
    check = await prov.run_node_command(
        command=["bash", "-c", " ; ".join([
            "grep -E 'HugePages_(Total|Free)|Hugepagesize' /proc/meminfo",
            f"ip -d link show {pf} | grep -E '^ +vf '",
            f"for vf in /sys/class/net/{pf}/device/virtfn*; do "
            "echo $(basename $(readlink $vf)) $(basename $(readlink $vf/driver)); done",
            "systemctl is-active ptp4l phc2sys",
            "/usr/local/sbin/pmc -u -b 0 -d 24 'GET TIME_STATUS_NP' | grep -E 'master_offset|gmPresent|gmIdentity'",
            "true",  # print every check, even when one fails
        ])],
        nodes=[node],
        become=True,
        quiet=True,
    )
    print(f"--- {node} ({pf})\n{check}")


In [ ]:
# The SR-IOV device plugin must advertise one VF per resource on each RAN node, and the kubelet the 1G hugepages
for node in FHI72_NODES:
    allocatable = await prov.run_node_command(
        command=["kubectl", "--kubeconfig", "/etc/kubernetes/admin.conf", "get", "node", node,
                 "-o", "jsonpath={.status.allocatable}"],
        nodes=["node1"],
        become=True,
        quiet=True,
    )
    print(f"{node}: {allocatable}")


## **3.**   5G core deployment:

In [ ]:
from ifran.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## **4.**   RAN deployment:

First, we are going to provision the remaining resources (RUs and UEs):

IFRAN powers on the UEs and the RUs (`benetel1`, `benetel2`) through the radio API, then waits until the API reports each RU ready. The API checks that an RU answers ping on its management network (192.168.233.0/24 for the Benetel RUs): the radio initialization of the RU is checked later by `ran.configure_rus()`.

> The commented code below is the former workaround, for an API that cannot reach the RU management network (it answered 503 "not pingeable" for the Benetel RUs until 2026-11-18): it powers the RUs on through the API, checks their management IP from their RAN node and marks them as provisioned. Use it instead of `provision()` if that happens again.

In [ ]:
prov_radio = ProvisioningManagement(core)
await prov_radio.provision()

# Former workaround, if the radio API cannot reach the RU management network again (see above):
# import json
# import urllib.request
#
# prov_radio = ProvisioningManagement(core)
# await prov_radio.provision_ues()
#
# BENETEL_RUS = {"benetel1": "sopnode-f3", "benetel2": "sopnode-f4"}  # RU: RAN node that reaches its management IP
# radio_api_params = prov_radio.deployment_model.provisioners["r2lab"].params
# radio_api_url = str(radio_api_params.base_url).rstrip("/")
#
#
# def radio_api(method, path, body=None, token=None):
#     headers = {"Content-Type": "application/json"} | ({"Authorization": f"Bearer {token}"} if token else {})
#     data = None if body is None else json.dumps(body).encode()
#     request = urllib.request.Request(radio_api_url + path, data=data, headers=headers, method=method)
#     with urllib.request.urlopen(request, timeout=300) as response:  # power on takes some time
#         return json.loads(response.read() or b"null")
#
#
# radio_token = radio_api_params.token or radio_api(
#     "POST", f"/token/{radio_api_params.token_params.infra}", radio_api_params.token_params.input_body
# )["access_token"]
# for ru_name, ran_node in BENETEL_RUS.items():
#     radio_api("POST", f"/power/{ru_name}/power_on", token=radio_token)
#     ru = prov_radio.deployment_model.rus[ru_name]
#     ru_ip = str(ru.ips[0])
#     await prov_radio.run_node_command(
#         command=["sh", "-c", f"for i in $(seq 60); do ping -c 1 -W 2 {ru_ip} >/dev/null && exit 0; sleep 5; done; exit 1"],
#         nodes=[ran_node],
#         quiet=True,
#     )
#     ru.is_provisioned = True
#     print(f"{ru_name} ({ru_ip}): powered on, reachable from {ran_node}")


### Then we are going to deploy the OCUDU CU-CP and CU-UP, and the first DU (`du01` with `benetel1`).

Each DU pod requests the two VFs (it uses the U-plane one) and the 1G hugepages of its node, and pins its threads on the CPUs allocated by the static CPU manager (`expert_execution` and DPDK EAL placeholders, replaced at start by the nador chart).

Before the DUs, `configure_rus()` loads the configuration of `benetel1` and `benetel2` (`/etc/ru_config.cfg`) from the RAN node of their DU, which reaches their management IP: DU MAC addresses and VLAN of the `fhi72` VFs of that node, centre frequency, bandwidth and TDD pattern of the DU radio configuration, MIMO mode of the RU ports, and the `benetel` parameters of the RU. An RU is rebooted only when its configuration changes (about 2 minutes).

In [ ]:
from ifran.deployment import RANManagement

ran = RANManagement(prov_radio)

In [ ]:
await ran.configure_rus()


In [ ]:
# Select the nador OCUDU charts: O-RAN 7.2 fronthaul of the gNB/DU (SR-IOV VF, 1G hugepages, VF and CPU
# placeholders replaced at start), CSI logger options.
from importlib.resources import files

ocudu_charts = files("nador") / "assets" / "ran" / "ocudu"
for name, component in ran.deployment_model.ran.components.items():
    if component.implementation == "ocudu" and component.type in ("gnb", "du"):
        component.custom_chart_path = str(ocudu_charts / f"ocudu-{component.type}")
        component.custom_rendered_files = [
            str(ocudu_charts / "values.yaml.j2"),
            str(ocudu_charts / "config.yaml.j2"),
        ]

### CSI logging

`du01` (`benetel1` on `sopnode-f3`) logs SRS channel estimates per resource block: the OCUDU CSI logger is in the SRS estimator of the PHY, which runs in the DU. `du02` could too, but `sopnode-f4` has no data disk for the CSI file (its disk holds a Proxmox installation), and the CU-CP and CU-UP have no PHY, hence no SRS. `enable_csi` selects the OCUDU DU image with the CSI logger (`r2labuser/ocudu-du-csi`, built from `ocudu-csi-logging` with `build_image.sh -c du`) for the radios of `CSI_COMPONENTS` only, adds periodic SRS to the rendered configuration, and sets the logger output to `/mnt/csi` on the node (the only hostPath of the OCUDU charts): each logger writes `/mnt/csi/csi_per_rb.csv`. Run these cells before preparing the radios.

`CSI_ENV` passes options to the logger (read when the radio starts), `CSI_COLLECT` sets how each experiment retrieves the CSI: see `examples/monitoring/multi-gnb-2du/test.ipynb` for the option tables.


In [ ]:
from nador.csi import CsiCollector, enable_csi

# CSI mode: radios that log (empty list: no CSI) and logger options (empty: defaults, RB granularity)
CSI_COMPONENTS = ['du01']  # du02: no data disk for the CSI file on sopnode-f4
CSI_ENV = {}  # e.g. {"CSI_GRANULARITY": "subcarrier", "CSI_SUBCARRIER_SAMPLING": "2"}
# CSI retrieval by each ExperimentRun: per-section files only (the whole file grows by several MB/s)
CSI_COLLECT = {'fetch_full': False, 'window_types': 'section', 'padding_seconds': 0, 'cpu_list': '0-4'}

enable_csi(ran, components=CSI_COMPONENTS, env=CSI_ENV)
for name in CSI_COMPONENTS:
    component = ran.deployment_model.ran.components[name]
    print(f"{name} on {component.node_name}: {component.image.repository}:{component.image.image_tag}",
          component.extra_env_vars)

# One collector per CSI component, reused by every ExperimentRun below.
CSI_COLLECTORS = [CsiCollector(ran, name, **CSI_COLLECT) for name in CSI_COMPONENTS]


The R2Lab nodes boot a live image: their root filesystem is in RAM, where the CSI file would fill the memory
(5g_ansible saw the gNB pod killed after about 2 minutes). `prepare_csi_storage` bind-mounts the `csi-data`
directory of the node's data disk onto `/mnt/csi`, as 5g_ansible does; the partitions are those of the 5g_ansible
inventory (`storage=`). The disk is never formatted, and the mount does not survive a reboot: rerun this cell after
reprovisioning. It stops with an explicit error if a node's directory is in RAM and no partition is given.

In [ ]:
from nador.csi import prepare_csi_storage

# Data disk partition (under /dev) of each radio node whose root is in RAM; nodes on a disk need no entry
CSI_STORAGE = {'sopnode-f3': 'sdb2'}  # 5g_ansible deploy.sh get_storage
for directory, report in ((await prepare_csi_storage(ran, CSI_STORAGE)) if CSI_COMPONENTS else {}).items():
    print(f"{directory}: {report['before']}  ->  {report['after']}")

In [ ]:
await ran.prepare(components=['cucp', 'cuup'])

In [ ]:
await ran.install(components=['cucp', 'cuup'])

In [ ]:
await ran.deploy(components=['du01'])

Check logs of `du01`:

In [ ]:
print(await ran.get_logs(quiet=True))

The DU is deployed and ready for connections.

## **5.**   Connecting UEs:

## Monitoring before UE attachment

Monitoring runs on the VM **node11**: the CPUs of the RAN nodes are isolated for the DUs. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement
from nador.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

### Live CSI view

One Streamlit view per CSI radio runs on `node11`, never on a radio node: it follows the radio node's CSI
file over SSH, where only `sshd` and `tail -F` run, at the lowest priority on the housekeeping CPUs `0-4`. The view
shows a sliding window (120 s by default) in local time. Start it once the radio runs (it reads the file header);
the tunnel is opened on this machine. `du02` has no CSI logger here (see above).

In [ ]:
from nador.csi import CsiLiveView
from IPython.display import Markdown, display

# Live view on node11, never on a radio node; housekeeping CPUs for "tail -F" on the radio node
CSI_LIVE = {"window_s": 120, "refresh_s": 10, "max_mb": 1024, "cpu_list": "0-4"}
CSI_LIVE_PORTS = {'du01': 8501, 'du02': 8502}
lives = globals().get("lives", {})
for name in [n for n in ['du01'] if n in CSI_COMPONENTS]:
    if name in lives:
        await lives[name].close_tunnel()  # re-run: free the local port of the previous tunnel
    lives[name] = CsiLiveView(ran, live_node='node11', component=name, port=CSI_LIVE_PORTS[name], **CSI_LIVE)
    url = await lives[name].start()
    display(Markdown(f"[CSI {name}]({url})"))

In [ ]:
from ifran.deployment import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
from nador.handover.ocudu import connected_ues

# Handover scenario: qhat01, then qhat02, on du01, one at a time; qhat01 is handed over (C-RNTI below).
HO_UES = ["qhat01", "qhat02"]
# qhat03 stays disconnected, as in the OAI notebook: turn its radio off
await ue.disconnect(ues=["qhat03"])
before = {u["ue"] for u in connected_ues(await ran.get_logs(components=["cucp"], quiet=True))}
await ue.connect(ues=["qhat01"])

Check that `qhat01` is attached to `du01` (the first DU of the CU-CP, `du=0`) and get its C-RNTI, before connecting `qhat02`:

In [ ]:
# UEs attached to the CU-CP: UE index, gNB-DU ID (du= of the F1AP log lines, gnb_du_id of the DU) and C-RNTI
DU01_ID = ran.deployment_model.ran.components["du01"].advanced_config.gnb_du_id
ues = connected_ues(await ran.get_logs(components=["cucp"], quiet=True))
for u in ues:
    print(f"UE {u['ue']}: du={u['du']}, C-RNTI {u['c_rnti']}")
new = [u for u in ues if u["ue"] not in before and u["du"] == DU01_ID]
if len(new) != 1:
    raise RuntimeError(f"expected one new UE on du01 for qhat01, found {new}: connect qhat01 again")
HO_RNTI = new[0]["c_rnti"]  # C-RNTI of qhat01 in the du01 cell, for the handover command

In [ ]:
await ue.connect(ues=["qhat02"])

Check `du01` logs after UEs were connected:

In [ ]:
print(await ran.get_logs(quiet=True))

`qhat01` and `qhat02` are connected to the 5G network.

Let's verify their connectivity by pinging the UPF:

In [ ]:
ping = await prov_radio.run_ue_command(
    command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], ues=HO_UES, quiet=True, flatten=False
)

In [ ]:
print(ping["qhat01"])

In [ ]:
print(ping["qhat02"])

## **6.**   Deploy the second DU (`du02` with `benetel2`):

In [ ]:
ran = RANManagement(ue)

In [ ]:
await ran.deploy(components=['du02'])

Check that each CSI logger runs and that its file grows on disk: the logs show the logger lines, and each
`csi_per_rb.csv` must grow between the two listings (the logger flushes every 5 seconds); `df` shows the space left.


In [ ]:
from ifran.deployment import ProvisioningManagement

for name in CSI_COMPONENTS:
    logs = await ran.get_logs(components=[name], quiet=True)
    print(f"--- {name}")
    print("\n".join(line for line in logs.splitlines() if "CSI" in line))

csi_nodes = {name: ran.deployment_model.ran.components[name].node_name for name in CSI_COMPONENTS}
nodes = ProvisioningManagement(preceding=ran)
for _ in range(2):  # sizes must grow between the two listings
    listing = await nodes.run_node_command(
        command=["sh", "-c", "ls -l /mnt/csi/csi_per_rb.csv; df -h /mnt/csi | tail -1"],
        nodes=sorted(set(csi_nodes.values())), become=True, quiet=True, flatten=False,
    )
    for name, node in csi_nodes.items():
        print(f"{name} ({node}):\n{listing[node].strip()}")
    await asyncio.sleep(10)

#### Check `du01`'s and `du02`'s logs before the F1-handover attempt:

In [ ]:
logs = await ran.get_logs(components=["du01", "du02"], quiet=True)

## **7.**   Trigger F1-Handover using the OCUDU CU-CP remote command:

UEs attached to the CU-CP, with their C-RNTI (`HO_RNTI` is the C-RNTI of `qhat01`, found after its attachment):

In [ ]:
ues = connected_ues(await ran.get_logs(components=["cucp"], quiet=True))
for u in ues:
    print(f"UE {u['ue']}: du={u['du']}, C-RNTI {u['c_rnti']}")
if not any(u["du"] == DU01_ID and u["c_rnti"] == HO_RNTI for u in ues):
    raise RuntimeError(f"C-RNTI {HO_RNTI} is no longer attached to du01: run the qhat01 cells again")

`run_handover` triggers the handover of the UE of C-RNTI `HO_RNTI` from `du01` to `du02` with the remote command `trigger_handover` (PCI of `du01`, C-RNTI, PCI, PLMN and TAC of `du02`) sent to the remote control server of the CU-CP (WebSocket, port 8001 of the CU-CP pod, `remote_control` in the CU-CP configuration), waits 15 s, then checks each step in the CU-CP and DU logs and explains it: decision of the CU-CP, preparation of the UE on `du02` (F1AP UE context setup, new C-RNTI), handover command to the UE through `du01`, random access on `du02` (dedicated preamble with contention-free access), `RRCReconfigurationComplete` received from `du02`, release of the UE on `du01`. It reports what followed a failure and the ping of the COTS UE.

The console command `ho` of the CU-CP is not used: on this OCUDU release it times out (`Dispatch trigger_handover timed out after 5s`). Part of the OCUDU log lines it looks for come from the srsRAN Project source (`nador.handover.ocudu`): if a step is reported missing while its line is in the logs, the patterns need adapting to this OCUDU release.

In [ ]:
from nador.handover import run_handover

HO_UE = "qhat01"  # COTS UE of HO_RNTI (ping after the handover)
handover = await run_handover(
    ran, HO_RNTI, source="du01", target="du02",
    ue_manager=ProvisioningManagement(preceding=ue), ue_name=HO_UE, ue_interface="wwan0",
)

Reference: https://docs.ocudu.org/tutorials/handover/ (handover between the cells of one CU-CP).

## Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline. The run also collects the CSI of each radio, cut per section under `csi/<component>/`.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun
from ifran.deployment import ProvisioningManagement
from ifran.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = HO_UES  # qhat03 is not connected in this scenario
UE_INTERFACE = "wwan0"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

server_node = monitoring.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname or SERVER_NODE
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(monitoring.deployment_model.params.ansible.runner_path) / "iperf-server")
clients = ProvisioningManagement(preceding=ue)

iperf_setup_error = None
try:
    await run_ansible_playbook(
        playbook=[
            {
                "name": "Install iperf3 on the server host",
                "hosts": SERVER_NODE,
                "gather_facts": False,
                "become": True,
                "tasks": [
                    {
                        "ansible.builtin.apt": {
                            "name": "iperf3",
                            "state": "present",
                            "update_cache": True,
                            "cache_valid_time": 3600,
                        }
                    }
                ],
            }
        ],
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )
    await clients.run_ue_command(
        command=[
            "sh",
            "-c",
            "command -v iperf3 >/dev/null 2>&1 || (DEBIAN_FRONTEND=noninteractive apt-get update && DEBIAN_FRONTEND=noninteractive apt-get install -y --no-upgrade iperf3)",
        ],
        ues=CLIENTS,
        become=True,
        quiet=True,
        flatten=False,
    )
except RuntimeError as exc:
    iperf_setup_error = str(exc)
    print(f"iperf setup failed; ping and monitoring checks will continue: {exc}")

addresses = await clients.run_ue_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], ues=CLIENTS, quiet=True, flatten=False
)
ue_ips = {}
for client in CLIENTS:
    ipv4 = [
        a["local"] for device in json.loads(addresses[client]) for a in device["addr_info"] if a["family"] == "inet"
    ]
    if len(ipv4) != 1:
        raise RuntimeError(f"{client}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[client] = ipv4[0]
print(f"Server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")

In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await clients.run_ue_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        ues=[client],
        become=True,
        quiet=True,
        flatten=False,
    )
    report = json.loads(output[client])
    if report.get("error"):
        raise RuntimeError(report["error"])
    return report


In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'qhat01', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'qhat01', "direction": "downlink", "duration": DURATION},
    {"name": "ue2_uplink", "client": 'qhat02', "direction": "uplink", "duration": DURATION},
    {"name": "ue2_downlink", "client": 'qhat02', "direction": "downlink", "duration": DURATION},
    {"name": "ue3_uplink", "client": 'qhat03', "direction": "uplink", "duration": DURATION},
    {"name": "ue3_downlink", "client": 'qhat03', "direction": "downlink", "duration": DURATION},
]
SCENARIO = [step for step in SCENARIO if step["client"] in CLIENTS]  # connected UEs only

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    csi=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    with experiment.section("ping"):
        ping = await clients.run_ue_command(
            command=["ping", "-c", "6", "11.1.1.1", "-I", UE_INTERFACE], ues=CLIENTS, quiet=True, flatten=False
        )
    experiment.save_json("ping.json", ping)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        try:
            if iperf_setup_error:
                raise RuntimeError(iperf_setup_error)
            await start_iperf_server(port)

            with experiment.section(test["name"]):
                report = await iperf_client(
                    client=test["client"],
                    direction=test["direction"],
                    duration=test["duration"],
                    port=port,
                )
        except (RuntimeError, ValueError) as exc:
            report = {"error": str(exc)}
            print(f"FAILED {test['name']}: {exc}")

        experiment.save_json(f"{test['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")


In [ ]:
from nador.artifacts import iperf_summary

print(iperf_summary(output_directory))

Check the CSI collected by the run: per radio, its node clock (the offset to this machine is only applied above
0.5 s), the rows and 5-second batches read in the whole file, and the per-section files under
`csi/<component>/by_window/section/`.

In [ ]:
meta = json.loads((output_directory / "experiment_metadata.json").read_text())
print(meta["status"], meta["collection_errors"])
for name, m in meta.get("csi", {}).items():  # empty without CSI
    print(f"--- {name} on {m['node']}")
    print("  clock:", {k: m.get(k) for k in ("clock_offset_applied", "clock_offset_applied_s", "clock_drift_s", "warnings")})
    print("  start:", {k: round(m["clock_start"][k], 3) for k in ("offset_s", "rtt_s", "uncertainty_s")})
    print("  data: ", {k: m.get("summary", {}).get(k) for k in ("rows", "markers", "dropped_rows", "windows_written", "windows_empty")})
print(sorted(str(p.relative_to(output_directory)) for p in (output_directory / "csi").rglob("*.gz")))

### End of the experiment

**Run All** stops at the next cell, so that the end-of-experiment cells below (stop of the live CSI views, UE
disconnection, release of the nodes) only run when you run them yourself, one at a time. The kernel keeps its state:
run the cells below with Shift+Enter when the experiment is over.

In [ ]:
class StopRunAll(Exception):
    """Ends "Run All" here; IPython shows only the message below (no traceback)."""

    def _render_traceback_(self):
        return [str(self)]


raise StopRunAll("Run All stops here. When the experiment is over, run the cells below one at a time.")

Stop the live CSI views before releasing the nodes: this closes the tunnels, stops the viewers and revokes their
SSH keys on the radio nodes.

In [ ]:
for view in globals().get("lives", {}).values():
    await view.stop()

Let's disconnect the UEs for now:

In [ ]:
ue = UEManagement(ran)
await ue.disconnect(force=True)

And finally save this experiment's meta model and deployment model:

In [ ]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/ifran/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.

## Cleanup — only after the experiment

Run the next cell **after** the experiment (and after stopping the live CSI views) to destroy the VMs. The POS nodes and the R2Lab resources are left as they are. The cell does nothing until you set `CONFIRM_VM_CLEANUP = True` in it: the VMs hold the Kubernetes control plane, the CUs, the core and its database, and the monitoring.

In [ ]:
CONFIRM_VM_CLEANUP = False  # set to True to destroy the VMs: this ends the experiment
if not CONFIRM_VM_CLEANUP:
    raise RuntimeError(f"VM cleanup not confirmed: set CONFIRM_VM_CLEANUP = True in this cell to destroy {VM_NODES}")
await ProvisioningManagement(preceding=ue).unprovision_nodes(nodes=VM_NODES)